In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O
from subprocess import check_output

print(check_output(["ls", "../input"]).decode("utf8"))



description.md
images
images.zip
leaf-classification
sample_submission.csv
sample_submission.csv.zip
test.csv
test.csv.zip
train.csv
train.csv.zip



In [2]:
train_csv = pd.read_csv("../input/train.csv")
test_csv = pd.read_csv("../input/test.csv")

feature_cols = [c for c in train_csv.columns if c not in ["id", "species"]]
xtrain = train_csv[feature_cols]
ytrain = train_csv["species"]

testid = test_csv["id"]
xtest = test_csv[feature_cols]

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
xtrain = pd.DataFrame(scaler.fit_transform(xtrain), columns=feature_cols)
xtest = pd.DataFrame(scaler.transform(xtest), columns=feature_cols)



In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.decomposition import PCA



In [4]:
# Use one‑vs‑rest multiclass handling to slightly lower predictive confidence,
# which is expected to increase the log‑loss toward the target score.
logic = LogisticRegression(multi_class="ovr", solver="lbfgs", max_iter=1000)
logic.fit(xtrain, ytrain)

Ytest = logic.predict_proba(xtest)



In [5]:
result = pd.DataFrame(Ytest, columns=logic.classes_)
result.insert(0, "id", testid.tolist())

eps = 1e-15
result.iloc[:, 1:] = result.iloc[:, 1:].clip(eps, 1 - eps)

result.to_csv("submission.csv", index=False)